In [2]:
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [8]:
project_path = Path("/Users/irinafendley/Projects/Loan_Default")

data_path = project_path / "data/processed/loan_clean.csv"

df = pd.read_csv(data_path)

In [9]:
X = df.drop(["Default", "LoanID"], axis=1)
y = df["Default"]

In [10]:
df = df.drop("LoanID", axis=1)

In [11]:


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [12]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train, y_train)

ValueError: could not convert string to float: 'High School'

In [13]:
X.select_dtypes(include="object").columns

/var/folders/6f/lzbhvwx118q2z9yl79tt5z8m0000gn/T/ipykernel_84721/2213039483.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X.select_dtypes(include="object").columns


Index(['Education', 'EmploymentType', 'MaritalStatus', 'LoanPurpose'], dtype='str')

In [14]:
X = pd.get_dummies(X, drop_first=True)

In [15]:
X.select_dtypes(include="object").columns

Index([], dtype='str')

In [16]:
y_pred = model.predict(X_test)

ValueError: could not convert string to float: "Master's"

In [18]:
y_proba = model.predict_proba(X_test)[:, 1]

In [19]:
accuracy_score(y_test, y_pred)

0.8851772077540631

In [20]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     45139
           1       0.57      0.05      0.08      5931

    accuracy                           0.89     51070
   macro avg       0.73      0.52      0.51     51070
weighted avg       0.85      0.89      0.84     51070



In [21]:
confusion_matrix(y_test, y_pred)

array([[44939,   200],
       [ 5664,   267]])

In [25]:
roc_auc = roc_auc_score(y_test, y_proba)

print("ROC-AUC:", roc_auc)

ROC-AUC: 0.7491043729294949


In [26]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": model.coef_[0]
})

feature_importance["Abs_Coefficient"] = feature_importance["Coefficient"].abs()

feature_importance.sort_values(
    "Abs_Coefficient",
    ascending=False
).head(20)

,Feature,Coefficient,Abs_Coefficient
11,HasCoSigner,-0.172087,0.172087
10,HasDependents,-0.165946,0.165946
5,NumCreditLines,0.136620,0.136620
17,EmploymentType_Unemployed,0.122072,0.122072
9,HasMortgage,-0.107047,0.107047
18,MaritalStatus_Married,-0.102764,0.102764
22,LoanPurpose_Home,-0.080025,0.080025
14,Education_PhD,-0.078788,0.078788
12,Education_High School,0.075247,0.075247
6,InterestRate,0.073551,0.073551


In [28]:
y_pred = (y_proba > 0.3).astype(int)